# Camada Bronze — TS_ALUNO (Microdados INEP)

**Pipeline:** Análise da Alfabetização no Brasil
**Responsabilidade:** ingerir o microdado de aluno individual (2.2M linhas) e disponibilizar em Parquet para a Silver.

**Por que um notebook separado:** o TS_ALUNO tem volume muito maior que as demais fontes (2.2M vs. no máximo ~24k linhas). Schema é definido manualmente — `inferSchema` nesse volume é lento e caro à toa, já que conhecemos as colunas.

**Escopo:** só as colunas de identificação e resultado (proficiência, alfabetização) seguem para a Gold. As colunas de item de prova (blocos, respostas, gabarito) ficam disponíveis na Silver, mas não são agregadas — são detalhe psicométrico fora do escopo do indicador de alfabetização.

## 1. Configuração de Acesso

Mesmo padrão dos demais notebooks — conexão ao ADLS Gen2 via Azure Key Vault.

In [1]:
# ============================================================
# Configuração de acesso ao ADLS Gen2 via Azure Key Vault
# ============================================================
from notebookutils import mssparkutils

CONTA_STORAGE = "stalfalfabetizacao"

CHAVE_ACESSO = mssparkutils.credentials.getSecret(
    "https://kv-alfabetizacao.vault.azure.net/",
    "storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{CONTA_STORAGE}.dfs.core.windows.net",
    CHAVE_ACESSO
)

CAMINHO_BRONZE = f"abfss://bronze@{CONTA_STORAGE}.dfs.core.windows.net/"

print("Conexão configurada com sucesso!")
print(f"  Bronze: {CAMINHO_BRONZE}")

StatementMeta(sparkpool, 12, 2, Finished, Available, Finished, False)

Conexão configurada com sucesso!
  Bronze: abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/


## 2. Ingestão do TS_ALUNO

Schema definido manualmente (27 colunas confirmadas no cabeçalho real do CSV). Sem `inferSchema` — com 2.2M linhas, inferir schema automaticamente é caro e desnecessário, já que conhecemos as colunas.

In [2]:
# ============================================================
# Leitura do TS_ALUNO com schema manual
# Volume alto (2.2M linhas) — schema explícito evita custo de inferSchema
# ============================================================
from pyspark.sql.types import *

schema_ts_aluno = StructType([
    StructField("NU_ANO_AVALIACAO",     IntegerType()),
    StructField("CO_UF",                IntegerType()),
    StructField("SG_UF",                StringType()),
    StructField("ID_ALUNO",             LongType()),
    StructField("TP_SERIE",             IntegerType()),
    StructField("ID_ESCOLA",            LongType()),
    StructField("TP_DEPENDENCIA",       IntegerType()),
    StructField("CO_MUNICIPIO",         LongType()),
    StructField("NO_MUNICIPIO",         StringType()),
    StructField("IN_PRESENCA_LP",       IntegerType()),
    StructField("IN_PREENCHIMENTO_LP",  IntegerType()),
    StructField("CO_CADERNO_LP",        IntegerType()),
    StructField("CO_BLOCO_1",           IntegerType()),
    StructField("TX_RESPOSTA_BLOCO_1",  StringType()),
    StructField("TX_GABARITO_BLOCO_1",  StringType()),
    StructField("CO_BLOCO_2",           IntegerType()),
    StructField("TX_RESPOSTA_BLOCO_2",  StringType()),
    StructField("TX_GABARITO_BLOCO_2",  StringType()),
    StructField("CO_BLOCO_3",           IntegerType()),
    StructField("TX_RESPOSTA_BLOCO_3",  StringType()),
    StructField("TX_GABARITO_BLOCO_3",  StringType()),
    StructField("CO_BLOCO_4",           IntegerType()),
    StructField("TX_RESPOSTA_BLOCO_4",  StringType()),
    StructField("TX_GABARITO_BLOCO_4",  StringType()),
    StructField("VL_PESO_ALUNO_LP",     DoubleType()),
    StructField("VL_PROFICIENCIA_LP",   DoubleType()),
    StructField("IN_ALFABETIZADO",      IntegerType()),
])

df_ts_aluno = spark.read.csv(
    CAMINHO_BRONZE + "TS_ALUNO.csv",
    header=True,
    schema=schema_ts_aluno,
    sep=";"
)

print(f"TS_ALUNO lido: {df_ts_aluno.count()} linhas")
df_ts_aluno.printSchema()

StatementMeta(sparkpool, 12, 3, Finished, Available, Finished, False)

TS_ALUNO lido: 2222792 linhas
root
 |-- NU_ANO_AVALIACAO: integer (nullable = true)
 |-- CO_UF: integer (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- ID_ALUNO: long (nullable = true)
 |-- TP_SERIE: integer (nullable = true)
 |-- ID_ESCOLA: long (nullable = true)
 |-- TP_DEPENDENCIA: integer (nullable = true)
 |-- CO_MUNICIPIO: long (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- IN_PRESENCA_LP: integer (nullable = true)
 |-- IN_PREENCHIMENTO_LP: integer (nullable = true)
 |-- CO_CADERNO_LP: integer (nullable = true)
 |-- CO_BLOCO_1: integer (nullable = true)
 |-- TX_RESPOSTA_BLOCO_1: string (nullable = true)
 |-- TX_GABARITO_BLOCO_1: string (nullable = true)
 |-- CO_BLOCO_2: integer (nullable = true)
 |-- TX_RESPOSTA_BLOCO_2: string (nullable = true)
 |-- TX_GABARITO_BLOCO_2: string (nullable = true)
 |-- CO_BLOCO_3: integer (nullable = true)
 |-- TX_RESPOSTA_BLOCO_3: string (nullable = true)
 |-- TX_GABARITO_BLOCO_3: string (nullable = true)
 |-- CO_BL

## 3. Gravação em Parquet

Converte para Parquet — mesma razão das demais camadas: formato colunar e comprimido, menor custo e leitura mais rápida na Silver.

In [3]:
# ============================================================
# Gravação do TS_ALUNO em Parquet na Bronze
# ============================================================

df_ts_aluno.write.mode("overwrite").parquet(CAMINHO_BRONZE + "ts_aluno/")

print("TS_ALUNO gravado em Parquet na Bronze:")
print(f"  {CAMINHO_BRONZE}ts_aluno/")

StatementMeta(sparkpool, 12, 4, Finished, Available, Finished, False)

TS_ALUNO gravado em Parquet na Bronze:
  abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/ts_aluno/
